# 14 - Robust (jittered / event-position-decorrelated) epoching

## Step 0 - Reusing the existing preprocessing pipeline

The epoching uses the raw annotade data, and applies the identical preprocessing steps from `11_preprocessing.ipynb`.


In [1]:
%load_ext autoreload
%autoreload 0

import json
import pathlib
import re
import sys
import time
from collections import Counter

import numpy as np
import pandas as pd
import mne

mne.set_log_level('WARNING')
pd.set_option('display.max_columns', 20)


In [2]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
COMPUTE_LIGHT = False

ANNOTATED_DATASET = pathlib.Path('data/annotated_dataset')  # persisted, fully-cleaned continuous EEG (Step 0)
EPOCH_DATASET = pathlib.Path('data/epoch_dataset')            # existing fixed-window epochs, used only for the Step 0 sanity check below
OUT_DIR = pathlib.Path('data/epoch_dataset_robust')           # new dataset written by this notebook
AR_OUT_DIR = pathlib.Path('data/epoch_dataset_robust_filtered')  # same dataset after AutoReject, stored separately

if COMPUTE_LIGHT:
    ANNOTATED_DATASET = pathlib.Path('data/annotated_dataset_light')  # persisted, fully-cleaned continuous EEG (Step 0)
    EPOCH_DATASET = pathlib.Path('data/epoch_dataset_light')            # existing fixed-window epochs, used only for the Step 0 sanity check below
    OUT_DIR = pathlib.Path('data/epoch_dataset_robust_light')           # new dataset written by this notebook
    AR_OUT_DIR = pathlib.Path('data/epoch_dataset_robust_light_filtered')  # same dataset after AutoReject, stored separately

SFREQ = 300.0                                   # confirmed from 11_preprocessing.ipynb (m['sample'] = timestamp * 300)
EPOCH_DURATION = 2.0                            # seconds -- same total duration as the existing fixed-window epochs
N_SAMPLES = int(round(EPOCH_DURATION * SFREQ)) + 1  # 601 samples, matching the existing 601-sample epochs
MARGIN = 0.2                                    # seconds -- minimum distance from the pinch to either epoch edge (Step 1)
SEED = 42                                       # fixed seed -> reproducible jitter draws
STEP3_STRIDE = 1.0                              # seconds -- Step 3 sliding-window stride (50% overlap for a 2.0s window)

# AutoReject (applied after the robust dataset is saved, see the "AutoReject" section at the end).
AR_N_INTERPOLATE = (1, 4, 8)                    # same candidate values as 12_data_quality_analysis.ipynb
AR_MIN_FIT_EPOCHS = 20                          # below this many non-overlapping epochs, fit on all epochs instead
AR_DECISION_L_FREQ = 1.0                        # Hz -- AutoReject decides on a copy with this extra highpass;
                                                # the saved epochs keep the dataset's own 0.3Hz highpass
AR_CONSENSUS = np.linspace(0.5, 1.0, 6)         # candidate fractions of bad channels needed to drop an epoch
                                                # (AutoReject default: np.linspace(0, 1, 11)); >= 0.5 -> only drop
                                                # an epoch if at least half its channels are bad

# Step 4: post-task finger-movement window (see the "Step 4" section below for the data evidence).
FINGER_MOVEMENT_MIN_SUBJECT_NUM = 33            # numeric subject-id threshold (>=), NOT a hardcoded "subject_35" string
FINGER_MOVEMENT_WINDOW_START_S = 65.0           # seconds before the end of the recording
FINGER_MOVEMENT_WINDOW_END_S = 5.0              # seconds before the end of the recording

# NB: the subject list is no longer hardcoded here -- see the "Subject & session discovery"
# section right after this, which lists whatever is actually present under data/raw.


def subject_num(subject: str) -> int:
    '''Numeric subject id, e.g. 'subject_35' -> 35. Used for the Step 4 subject-id comparison,
    intentionally not a hardcoded 'subject_35' string match, so it naturally covers any later
    subjects with numeric id >= FINGER_MOVEMENT_MIN_SUBJECT_NUM too.'''
    return int(subject.split('_')[1])


rng = np.random.default_rng(SEED)

print(f"Epoch length: {EPOCH_DURATION}s = {N_SAMPLES} samples at {SFREQ}Hz")

Epoch length: 2.0s = 601 samples at 300.0Hz


## Loading: only the leaves we need, from the persisted `eeg` (fully cleaned) + `markers`


In [3]:
def load_manifest():
    with open(ANNOTATED_DATASET / 'manifest.json') as f:
        return json.load(f)


def load_session(manifest, subject, session):
    '''Load only the 'eeg' (fully-cleaned, annotated continuous Raw) and 'markers' leaves for one
    subject/session, instead of read_raw_dataset()'s default of loading every leaf (eeg_light,
    eeg_filtered, uncleaned eeg_raw, gaze, ...). This keeps I/O fast: ~0.05-0.2s per session
    instead of pulling in ~200MB of raw data we don't need.'''
    leaves = manifest[subject][session]
    sdir = ANNOTATED_DATASET / subject / session
    eeg = mne.io.read_raw_fif(sdir / leaves['eeg']['file'], preload=True)
    markers = pd.read_parquet(sdir / leaves['markers']['file'])
    return eeg, markers


manifest = load_manifest()
_eeg_probe, _markers_probe = load_session(manifest, 'subject_21', 'session_1')
print(f"sfreq={_eeg_probe.info['sfreq']}, n_channels={len(_eeg_probe.ch_names)}, "
      f"duration={_eeg_probe.times[-1]:.1f}s, markers shape={_markers_probe.shape}")
assert _eeg_probe.info['sfreq'] == SFREQ
del _eeg_probe, _markers_probe


sfreq=300.0, n_channels=21, duration=1526.5s, markers shape=(1579, 51)


## Subject & session discovery, and structural validation

Subjects are discovered by listing `data/raw` for directories matching `subject_<number>`

Sessions are likewise discovered per subject from `data/annotated_dataset/manifest.json` - some subjects only completed one session.

The `FINGER_MOVEMENT_MIN_SUBJECT_NUM >= 35` comparison from Step 4 is a plain numeric comparison on whatever subjects end up discovered here, so it needs no change to keep applying correctly to any newly-included subject with numeric id >= 35.

In [4]:
RAW_DIR = pathlib.Path('data/raw')
SUBJECT_PATTERN = re.compile(r'^subject_(\d+)$')

REFERENCE_CH_NAMES = ['P3', 'C3', 'F3', 'Fz', 'F4', 'C4', 'P4', 'Cz', 'A1', 'Fp1', 'Fp2',
                       'T7', 'P7', 'O1', 'O2', 'F7', 'F8', 'A2', 'P8', 'T8', 'Pz']
REFERENCE_SFREQ = 300.0
REQUIRED_MARKER_COLUMNS = {'timestamp', 'EventType', 'PinchType', 'EventName',
                            'ExperimentIndex', 'feedbackIsDelayed'}
EXPECTED_FREE_VAR_COLUMNS = {'hasVisualFeedback', 'hasAuditoryFeedback', 'hasVirtualHand',
                              'pinchIsHinted', 'isChoiceStimuli', 'numberOfStimuli',
                              'feedbackIsDelayed'}

# ---- 1) discover every subject_<N> directory actually present under data/raw ----
raw_entries = sorted(RAW_DIR.iterdir()) if RAW_DIR.exists() else []
discovered_subjects, excluded_raw_entries = [], []
for entry in raw_entries:
    match = SUBJECT_PATTERN.match(entry.name)
    if entry.is_dir() and match:
        discovered_subjects.append(entry.name)
    else:
        excluded_raw_entries.append(entry.name)
discovered_subjects = sorted(discovered_subjects, key=subject_num)

print(f"Found {len(discovered_subjects)} subject directories under data/raw: {discovered_subjects}")
print("Ignored non-subject entries under data/raw: "
      f"{excluded_raw_entries}" if excluded_raw_entries else
      "No non-subject entries found under data/raw.")

# ---- 2) cross-check against the persisted, cleaned data/annotated_dataset ----
subjects_no_cleaned_data = [s for s in discovered_subjects if s not in manifest]
subjects_with_cleaned_data = [s for s in discovered_subjects if s in manifest]
if subjects_no_cleaned_data:
    print(f"\nSKIPPING (no cleaned entry in data/annotated_dataset): {subjects_no_cleaned_data}")
else:
    print("\nEvery subject found under data/raw has a corresponding cleaned entry in "
          "data/annotated_dataset -- none skipped for this reason.")


def validate_subject_session(eeg: mne.io.BaseRaw, markers: pd.DataFrame) -> list[str]:
    '''Structural checks this notebook's logic actually depends on. Returns a list of failure
    reasons (empty = OK).'''
    reasons = []
    if float(eeg.info['sfreq']) != REFERENCE_SFREQ:
        reasons.append(f"sfreq={eeg.info['sfreq']} (expected {REFERENCE_SFREQ})")
    if list(eeg.ch_names) != REFERENCE_CH_NAMES:
        reasons.append(f"channel layout differs from the expected {len(REFERENCE_CH_NAMES)}-channel "
                        f"layout (got {len(eeg.ch_names)} channels: {list(eeg.ch_names)})")
    missing_core = REQUIRED_MARKER_COLUMNS - set(markers.columns)
    if missing_core:
        reasons.append(f"markers missing required columns: {sorted(missing_core)}")
    return reasons


# ---- 3) per-subject/session structural validation; discover sessions from the manifest ----
skip_log = []
SESSIONS_BY_SUBJECT = {}
for subject in subjects_with_cleaned_data:
    sessions_here = sorted(manifest[subject].keys())
    accepted = []
    for session in sessions_here:
        try:
            eeg, markers = load_session(manifest, subject, session)
        except Exception as e:
            skip_log.append((subject, session, f"failed to load: {e}"))
            continue
        reasons = validate_subject_session(eeg, markers)
        if reasons:
            skip_log.append((subject, session, '; '.join(reasons)))
            del eeg, markers
            continue
        missing_free_vars = EXPECTED_FREE_VAR_COLUMNS - set(markers.columns)
        if missing_free_vars:
            print(f"  [WARN] {subject}/{session}: missing optional free-variable columns "
                  f"{sorted(missing_free_vars)} -- proceeding anyway (that metadata will just be "
                  f"absent for this subject/session)")
        accepted.append(session)
        del eeg, markers
    if accepted:
        SESSIONS_BY_SUBJECT[subject] = accepted
    else:
        skip_log.append((subject, None, "no sessions passed structural validation"))

SUBJECTS = sorted(SESSIONS_BY_SUBJECT.keys(), key=subject_num)
n_sessions_total = sum(len(v) for v in SESSIONS_BY_SUBJECT.values())

print(f"\n{len(SUBJECTS)} subjects, {n_sessions_total} sessions total will be processed:")
for subject in SUBJECTS:
    print(f"  {subject}: {SESSIONS_BY_SUBJECT[subject]}")

if skip_log:
    print("\nSkipped (structural validation / no cleaned data), with reasons:")
    for s, ses, reason in skip_log:
        print(f"  {s}" + (f"/{ses}" if ses else "") + f": {reason}")
else:
    print("\nNo (subject, session) pairs were skipped for structural-validation reasons.")

print(f"\nStep 4 applies to subjects with numeric id >= {FINGER_MOVEMENT_MIN_SUBJECT_NUM}: "
      f"{[s for s in SUBJECTS if subject_num(s) >= FINGER_MOVEMENT_MIN_SUBJECT_NUM]}")


Found 19 subject directories under data/raw: ['subject_21', 'subject_22', 'subject_23', 'subject_24', 'subject_25', 'subject_26', 'subject_27', 'subject_28', 'subject_29', 'subject_30', 'subject_32', 'subject_33', 'subject_34', 'subject_35', 'subject_36', 'subject_37', 'subject_38', 'subject_39', 'subject_40']
No non-subject entries found under data/raw.

Every subject found under data/raw has a corresponding cleaned entry in data/annotated_dataset -- none skipped for this reason.

19 subjects, 36 sessions total will be processed:
  subject_21: ['session_1', 'session_2']
  subject_22: ['session_1', 'session_2']
  subject_23: ['session_1', 'session_2']
  subject_24: ['session_1', 'session_2']
  subject_25: ['session_1', 'session_2']
  subject_26: ['session_1', 'session_2']
  subject_27: ['session_1', 'session_2']
  subject_28: ['session_1', 'session_2']
  subject_29: ['session_1', 'session_2']
  subject_30: ['session_1', 'session_2']
  subject_32: ['session_1', 'session_2']
  subject_33

## Marker cleaning

`clean_markers` reproduces the duplicate-pinch-activation dedup (events <0.3s apart, keep the
first) and the delayed-feedback timestamp correction (-0.6s, stimulus pinches only), then `all_pinches` selects every
pinch event (`EventType=='PinchLog'`, all `PinchType`s). The narrower ground-truth positive class
used elsewhere in this repo (`PinchType=='PINCH'`) can be recovered from the metadata.


In [5]:
def clean_markers(m: pd.DataFrame) -> pd.DataFrame:
    '''Reproduce the marker-cleaning logic from 11_preprocessing.ipynb's epoching cell: drop
    duplicate pinch activations (<0.3s apart, keep first) and shift the timestamp of
    delayed-feedback stimulus pinches back by the 0.6s feedback delay.'''
    m = m.sort_values('timestamp').reset_index(drop=True)
    is_pinch = m['EventType'] == 'PinchLog'
    pinch_df = m[is_pinch].copy()
    gap = pinch_df['timestamp'].diff()
    new_group = (gap > 0.3) | gap.isna()
    pinch_df['group'] = new_group.cumsum()
    pinch_first = pinch_df.drop_duplicates(subset='group', keep='first').drop(columns='group')
    m = pd.concat([m[~is_pinch], pinch_first]).sort_values('timestamp').reset_index(drop=True)

    # Recomputed AFTER the dedup/reset above -- see the Step 0 markdown for why this differs
    # (deliberately) from 11_preprocessing.ipynb's literal code.
    is_pinch = m['EventType'] == 'PinchLog'
    # only stimulus selections are delayed (PinchHandler.feedbackDelay = 0.6s); PINCH_NONE / ignored pinches are logged at once
    is_delayed = m['feedbackIsDelayed'].fillna(False).astype(bool) & (m['EventName'] == 'PINCH_STIMULUS')
    m.loc[(is_pinch & is_delayed), 'timestamp'] -= 0.6
    m = m.sort_values('timestamp').reset_index(drop=True)
    return m


def all_pinches(m: pd.DataFrame) -> pd.DataFrame:
    '''Every pinch event, i.e. all PinchLog rows regardless of PinchType (PINCH, GAZE,
    PINCH_WIZARD_OF_OZ, PINCH_IGNORED, ...). The original PinchType is kept in the metadata, so the
    `PinchType == 'PINCH'` ground-truth subset used by 12/14 can still be recovered by filtering.'''
    return m[m.EventType == 'PinchLog'].reset_index(drop=True)


## Valid time budget: a "bad time" mask from the pipeline's own annotations

`data/annotated_dataset`'s `eeg` already carries `BAD_BREAK` (inter-block breaks, tutorial, and
the end-of-recording tail -- computed by `mark_breaks_as_bad` in `11_preprocessing.ipynb`) and
`BAD_muscle` (muscle-artifact z-score annotation) annotations. `build_bad_mask` turns those into a
per-sample boolean array (`True` = inside a `BAD_*` annotation), which Steps 2 and 3 use to avoid
placing any window on bad-quality or non-experiment time.

In [7]:
def build_bad_mask(raw: mne.io.BaseRaw) -> np.ndarray:
    '''Per-sample boolean mask (local to `raw`, i.e. 0-based from raw's own t=0), True wherever
    a BAD_* annotation (BAD_BREAK, BAD_muscle) covers that sample.'''
    n = raw.n_times
    sfreq = raw.info['sfreq']
    mask = np.zeros(n, dtype=bool)
    for onset, duration, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        if desc.upper().startswith('BAD'):
            s0 = max(0, int(np.floor(onset * sfreq)))
            s1 = min(n, int(np.ceil((onset + duration) * sfreq)))
            mask[s0:s1] = True
    return mask


_eeg_chk2, _markers_chk2 = load_session(manifest, 'subject_21', 'session_1')
_bad_mask_chk = build_bad_mask(_eeg_chk2)
print(f"subject_21/session_1: {_bad_mask_chk.mean()*100:.1f}% of the recording is inside a BAD_* "
      f"annotation ({_bad_mask_chk.sum()/SFREQ:.0f}s of {_eeg_chk2.times[-1]:.0f}s)")
del _eeg_chk2, _markers_chk2, _bad_mask_chk


subject_21/session_1: 12.9% of the recording is inside a BAD_* annotation (197s of 1526s)


## Step 1: jittered pinch epochs

For every pinch (any `PinchLog` subtype), draw `t_offset ~ Uniform[MARGIN, EPOCH_DURATION - MARGIN]` =
`Uniform[0.2, 1.8]` with the fixed, seeded `rng = np.random.default_rng(42)` created above (shared
across all subjects/sessions, consumed in subject -> session -> pinch order, so the whole notebook
is reproducible end to end). The epoch window is then `[t_pinch - t_offset, t_pinch - t_offset + 2.0]`.

Implementation note: rather than passing a different `tmin`/`tmax` per event into `mne.Epochs`
(which only accepts one shared `tmin`/`tmax` for the whole call), each epoch is built from a
*phantom* "epoch-start" event placed at sample `round((t_pinch - t_offset) * sfreq)`, and
`mne.Epochs` is called once per session with a fixed `tmin=0.0, tmax=2.0` relative to that phantom
event. `t_offset` (the true pinch position within the epoch) is carried purely as metadata.
`reject_by_annotation=True` (the `mne.Epochs` default, passed explicitly here for clarity) means an
epoch touching a `BAD_BREAK`/`BAD_muscle` annotation is dropped automatically by MNE -- the same
behaviour the existing fixed-window `epoch_dataset` relies on -- so Step-1 windows are not
separately checked against the bad-time mask before being handed to `mne.Epochs`.

If a drawn `t_offset` would push the window outside the recording (only possible for pinches
extremely close to the very start/end of the session), up to 50 redraws are attempted before
giving up on that pinch (`step1_unfit`, reported per session below); in practice this never
triggers, across any session.


In [8]:
def sample_step1(pinches: pd.DataFrame, t_last: float, rng: np.random.Generator):
    '''Jittered pinch windows. Returns (rows, window_list) where `rows` are metadata dicts and
    `window_list` is the list of (s0, s1) sample ranges actually placed.'''
    rows = []
    windows = []
    n_unfit = 0
    for _, prow in pinches.iterrows():
        t_pinch = prow['timestamp']
        placed = False
        t_offset = None
        for _attempt in range(50):
            t_offset = rng.uniform(MARGIN, EPOCH_DURATION - MARGIN)
            tmin_local = t_pinch - t_offset
            tmax_local = tmin_local + EPOCH_DURATION
            if tmin_local >= 0 and tmax_local <= t_last:
                placed = True
                break
        if not placed:
            n_unfit += 1
            continue
        s0 = int(round(tmin_local * SFREQ))
        s1 = s0 + N_SAMPLES
        windows.append((s0, s1))
        row = prow.to_dict()
        row.update(dict(
            EventType='PinchLog',
            t_offset=t_offset, is_overlapping=False,
            overlap_neighbor_ids='', n_pinches_inside=1,
            step=1, s0=s0, s1=s1,
        ))
        rows.append(row)
    return rows, windows, n_unfit


## Step 2: non-overlapping `Empty` epochs

Sampling uses a single per-session occupancy bitmask at sample resolution (`forbidden`), seeded
with the bad-time mask, every pinch onset sample (any type), and every Step-1 window. A candidate
start sample is drawn uniformly at random from the valid range (from the first `EXPERIMENT_START`
to `n_samples - N_SAMPLES`); it is accepted only if its full 601-sample span is entirely clear of
`forbidden`, after which that span is marked occupied too -- so two accepted Step-2 windows, or a
Step-2 window and any Step-1 window, can never overlap, and no Step-2 window can ever contain a
pinch onset or touch a `BAD_*` segment. Because Step-2 windows never touch `BAD_*` time,
`reject_by_annotation` never drops them later (unlike a handful of Step-1 windows).

Target count: **one `Empty` epoch per Step-1 (pinch) epoch in that session** (a 1:1 ratio,
matching "roughly matching or somewhat exceeding" the Step-1 count from the brief). Rejection
sampling is capped at `max(20000, 400 * target)` attempts per session; if that cap is hit before
reaching the target, `step2_exhausted=True` is reported and the notebook does **not** silently
relax the non-overlap constraint -- it simply stops short and reports the shortfall. In practice
(see the per-session log below) every processed session reaches its target exactly, with
ample headroom (each session has, very roughly, 1000-1500s of valid non-bad experiment time, and
even 300 pinch epochs of 2s only "use up" ~600s of that when packed, so 2s windows are not a tight
resource here).


In [9]:
def sample_step2(bad_mask, pinch_mask, step1_windows, first_valid_sample, max_start_sample,
                  n_target, rng: np.random.Generator):
    n_local = len(bad_mask)
    occupied = np.zeros(n_local, dtype=bool)
    for (s0, s1) in step1_windows:
        occupied[s0:s1] = True
    forbidden_base = bad_mask | pinch_mask

    rows = []
    max_attempts = max(20000, n_target * 400)
    n_placed = 0
    attempts = 0
    while n_placed < n_target and attempts < max_attempts:
        attempts += 1
        s0 = int(rng.integers(first_valid_sample, max_start_sample + 1))
        s1 = s0 + N_SAMPLES
        if occupied[s0:s1].any() or forbidden_base[s0:s1].any():
            continue
        occupied[s0:s1] = True
        rows.append(dict(
            PinchType='Empty', EventType='NoPinchLog',
            t_offset=np.nan, is_overlapping=False,
            overlap_neighbor_ids='', n_pinches_inside=0,
            step=2, s0=s0, s1=s1,
        ))
        n_placed += 1
    exhausted = attempts >= max_attempts and n_placed < n_target
    return rows, occupied, exhausted


## Step 3: dense, deliberately-overlapping epochs

A sliding window at a **1.0s stride** (50% overlap for a 2.0s window -- dense enough to be useful
for augmentation without exploding dataset size/runtime) is swept across every valid sample
position in the session (from the first `EXPERIMENT_START` to `n_samples - N_SAMPLES`). A
candidate window is **skipped** if it touches any `BAD_*` annotation (data quality still matters
even for the augmentation pool), but unlike Steps 1-2 it is allowed to freely overlap Step-1,
Step-2, and other Step-3 windows -- that permissiveness is exactly what `is_overlapping=True`
flags.

Each Step-3 window is labelled by literally checking which pinches of any type (if any) have their
onset timestamp inside `[tmin, tmax]` (inclusive both ends): zero -> `Empty`; one or more -> the
pinch closest to the window's center is used for the label and metadata, and `n_pinches_inside`
records how many were found (so multi-pinch windows are auditable rather than silently
resolved). No additional random draws were layered on top of the sliding window -- at this stride
it already provides dense, even coverage of the valid time budget.


In [10]:
def sample_step3(bad_mask, pinches: pd.DataFrame, first_valid_sample, max_start_sample, m_columns):
    stride_samples = int(round(STEP3_STRIDE * SFREQ))
    empty_row_template = {c: np.nan for c in m_columns}
    rows = []
    for s0 in range(first_valid_sample, max_start_sample + 1, stride_samples):
        s1 = s0 + N_SAMPLES
        if bad_mask[s0:s1].any():
            continue
        tmin_local = s0 / SFREQ
        tmax_local_incl = (s0 + N_SAMPLES - 1) / SFREQ  # inclusive timestamp of the last sample

        inside = pinches[(pinches['timestamp'] >= tmin_local) & (pinches['timestamp'] <= tmax_local_incl)]
        n_inside = len(inside)
        if n_inside > 0:
            center = (tmin_local + tmax_local_incl) / 2.0
            idx = (inside['timestamp'] - center).abs().idxmin()
            prow = inside.loc[idx]
            row = prow.to_dict()
            row.update(dict(EventType='PinchLog', t_offset=prow['timestamp'] - tmin_local))
        else:
            row = dict(empty_row_template)
            row.update(dict(PinchType='Empty', EventType='NoPinchLog', t_offset=np.nan))

        row.update(dict(
            is_overlapping=True, overlap_neighbor_ids='', n_pinches_inside=n_inside,
            step=3, s0=s0, s1=s1,
        ))
        rows.append(row)
    return rows


## Overlap-neighbor bookkeeping (Step 3 only)

After assembling and sorting the whole session pool by start sample, an `O(n log n)` sweep (sort
by start, then for every `is_overlapping` row binary-search the range of rows whose start sample
falls before this row's end sample, and keep those whose own end sample is after this row's
start) finds every other epoch -- from *any* step -- that a given Step-3 epoch's window overlaps.
The resulting indices (final row positions in the saved `Epochs` object / metadata table) are
stored as a comma-separated string in `overlap_neighbor_ids`. Step-1 and Step-2 rows keep
`overlap_neighbor_ids=''` (empty) by construction, since the whole point of Steps 1-2 is that they
never overlap anything in that pool -- though a Step-3 window can, and typically does, overlap
them; that relationship is recorded from the Step-3 side.


In [11]:
def compute_overlap_neighbors(df: pd.DataFrame) -> pd.Series:
    s0_arr = df['s0'].to_numpy()
    s1_arr = df['s1'].to_numpy()  # df is already sorted by s0
    is_overlapping_arr = df['is_overlapping'].to_numpy()
    neighbor_lists = [[] for _ in range(len(df))]
    for i in range(len(df)):
        if not is_overlapping_arr[i]:
            continue
        hi = np.searchsorted(s0_arr, s1_arr[i], side='left')
        for j in range(hi):
            if j == i:
                continue
            if s1_arr[j] > s0_arr[i]:
                neighbor_lists[i].append(int(j))
    return pd.Series([','.join(map(str, lst)) for lst in neighbor_lists], index=df.index)


## Step 4: post-task finger-movement epochs (subject-conditional)

**Data evidence.** For subjects with numeric id >= 33, the last part of the recording is *not*
task time: it is a real, appended post-task period of unscripted, unlabelled finger movement.

This step **drops** any tile that contains a genuine pinch onset (its `FingerMovement` label would be
  wrong; `subject_39/session_1` has one `PINCH` at t=1319.75s inside its first tile), and


**What gets built.** For each qualifying session, 30 back-to-back 2.0s tiles are cut starting at
`t_last - 65s` (`t_last` = the timestamp of the recording's actual last sample), i.e. tile *i*
spans `[t_last-65+2i, t_last-63+2i]` for `i=0..29`, exactly filling the 60s window with no gap and
no overlap (adjacent tiles share only the single boundary instant where one epoch's last sample
equals the next epoch's first sample -- an inherent, negligible characteristic of any back-to-back
tiling of inclusive-endpoint MNE epochs, not a substantive overlap; `is_overlapping=False` for all
of them, matching the requirement that they don't overlap each other or anything else).

In [12]:
def sample_step4(eeg: mne.io.BaseRaw, subject: str, pinches: pd.DataFrame, m_columns) -> tuple[list[dict], int]:
    '''30 back-to-back, non-overlapping 2.0s 'FingerMovement' tiles from the last
    [FINGER_MOVEMENT_WINDOW_START_S, FINGER_MOVEMENT_WINDOW_END_S] seconds of the recording,
    for subjects with numeric id >= FINGER_MOVEMENT_MIN_SUBJECT_NUM. Returns ([], 0) for every other
    subject.

    Returns (rows, n_dropped_pinch): tiles containing a genuine pinch onset are dropped (their
    'FingerMovement' label would be wrong), and counted in n_dropped_pinch. Overlap with task-block
    time itself is allowed -- see check_step4_conflicts / process_session.'''
    if subject_num(subject) < FINGER_MOVEMENT_MIN_SUBJECT_NUM:
        return [], 0

    t_last = eeg.times[-1]
    win_start = t_last - FINGER_MOVEMENT_WINDOW_START_S
    win_end = t_last - FINGER_MOVEMENT_WINDOW_END_S
    if win_start < 0:
        return [], 0  # defensive: session too short for this window (not expected in practice)

    stride_samples = int(round(EPOCH_DURATION * SFREQ))  # 600 -- back-to-back, no gap/overlap
    n_tiles = int(np.floor((win_end - win_start) / EPOCH_DURATION))  # 30 for an exact 60s window
    s0_start = int(round(win_start * SFREQ))

    empty_row_template = {c: np.nan for c in m_columns}
    pinch_t = pinches['timestamp'].to_numpy()
    rows = []
    n_dropped_pinch = 0
    for i in range(n_tiles):
        s0 = s0_start + i * stride_samples
        s1 = s0 + N_SAMPLES
        # same inclusive [first sample, last sample] convention as Step 3's pinch check
        if ((pinch_t >= s0 / SFREQ) & (pinch_t <= (s1 - 1) / SFREQ)).any():
            n_dropped_pinch += 1
            continue
        row = dict(empty_row_template)
        row.update(dict(
            PinchType='FingerMovement', EventType='NoPinchLog',
            t_offset=np.nan, is_overlapping=False,
            overlap_neighbor_ids='', n_pinches_inside=0,
            step=4, s0=s0, s1=s1,
        ))
        rows.append(row)
    return rows, n_dropped_pinch


def check_step4_conflicts(step4_rows: list[dict], other_windows_df: pd.DataFrame) -> list[int]:
    '''Returns indices (into step4_rows) of any Step-4 candidate whose [s0, s1) sample range
    intersects an already-placed Step-1/2/3 window. This happens when the recording ends shortly
    after the last block, so the [t_last-65, t_last-5] window reaches back into task time (e.g.
    subject_39/session_1). That is acceptable; such tiles are flagged is_overlapping=True in
    process_session so the `is_overlapping == False` subset stays genuinely non-overlapping.'''
    if not step4_rows or len(other_windows_df) == 0:
        return []
    s0_4 = np.array([r['s0'] for r in step4_rows])
    s1_4 = np.array([r['s1'] for r in step4_rows])
    s0_o = other_windows_df['s0'].to_numpy()
    s1_o = other_windows_df['s1'].to_numpy()
    conflicts = []
    for i in range(len(s0_4)):
        if ((s0_o < s1_4[i]) & (s1_o > s0_4[i])).any():
            conflicts.append(i)
    return conflicts


## Assembling one session's `Epochs` object



In [13]:
def process_session(subject, session, eeg, markers, rng: np.random.Generator, verbose=True,
                    decision_eeg: mne.io.BaseRaw = None):
    t0 = time.time()
    n_local = eeg.n_times
    t_last = eeg.times[-1]

    m = clean_markers(markers)
    pinches = all_pinches(m)
    bad_mask = build_bad_mask(eeg)

    exp_start = markers[(markers.EventName == 'EXPERIMENT_START') & (markers.ExperimentIndex == 1)].timestamp.min()
    first_valid_sample = max(0, int(np.floor(exp_start * SFREQ))) if pd.notna(exp_start) else 0

    pinch_onset_samples = np.round(pinches['timestamp'].to_numpy() * SFREQ).astype(int)
    pinch_onset_samples = pinch_onset_samples[(pinch_onset_samples >= 0) & (pinch_onset_samples < n_local)]
    pinch_mask = np.zeros(n_local, dtype=bool)
    pinch_mask[pinch_onset_samples] = True

    max_start_sample = n_local - N_SAMPLES
    if max_start_sample <= first_valid_sample:
        raise RuntimeError(f"{subject}/{session}: session too short for any {EPOCH_DURATION}s epoch")

    # ---- Step 1 ----
    step1_rows, step1_windows, step1_unfit = sample_step1(pinches, t_last, rng)

    # ---- Step 2 ----
    step2_rows, _occupied, step2_exhausted = sample_step2(
        bad_mask, pinch_mask, step1_windows, first_valid_sample, max_start_sample,
        n_target=len(step1_windows), rng=rng,
    )

    # ---- Step 3 ----
    step3_rows = sample_step3(bad_mask, pinches, first_valid_sample, max_start_sample, m.columns)

    # ---- assemble Steps 1-3 ----
    # step1/step3-with-a-pinch rows already carry the full marker row; step2 (and step3-without-a-
    # pinch, handled inside sample_step3) rows need the NaN template merged in here.
    empty_row_template = {c: np.nan for c in m.columns}
    step2_rows_full = []
    for row in step2_rows:
        full = dict(empty_row_template)
        full.update(row)
        step2_rows_full.append(full)

    df123 = pd.DataFrame(step1_rows + step2_rows_full + step3_rows).sort_values('s0').reset_index(drop=True)

    # ---- Step 4 (subject-conditional: numeric subject id >= FINGER_MOVEMENT_MIN_SUBJECT_NUM) ----
    step4_rows, step4_dropped_pinch = sample_step4(eeg, subject, pinches, m.columns)
    step4_conflicts = check_step4_conflicts(step4_rows, df123)
    # The finger-movement window may reach back into task time (overlapping Step-1/2/3 windows).
    # That's fine -- flag those tiles as overlapping so the clean subset stays non-overlapping;
    # compute_overlap_neighbors below then records which epochs they overlap.
    for i in step4_conflicts:
        step4_rows[i]['is_overlapping'] = True

    if step4_rows:
        df = pd.concat([df123, pd.DataFrame(step4_rows)], ignore_index=True).sort_values('s0').reset_index(drop=True)
    else:
        df = df123
    df['epoch_id'] = np.arange(len(df))
    df['subject'] = subject
    df['session'] = session
    df['overlap_neighbor_ids'] = compute_overlap_neighbors(df)

    def _events_for(sub_df):
        sample_abs = sub_df['s0'].to_numpy() + eeg.first_samp
        events = np.column_stack([sample_abs, np.zeros_like(sample_abs),
                                   np.where(sub_df['EventType'] == 'PinchLog', 1, 0)]).astype(int)
        uniq, counts = np.unique(events[:, 0], return_counts=True)
        dup_vals = uniq[counts > 1]
        while len(dup_vals) > 0:  # mirrors 11_preprocessing.ipynb's own duplicate-sample handling
            edf = pd.DataFrame(events)
            offset = edf.groupby(0).cumcount()
            edf[0] = edf[0] + offset
            events = edf.to_numpy()
            uniq, counts = np.unique(events[:, 0], return_counts=True)
            dup_vals = uniq[counts > 1]
        return events

    meta_cols = [c for c in df.columns if c not in ('s0', 's1')]
    df_123_final = df[df['step'] != 4].reset_index(drop=True)
    df_4_final = df[df['step'] == 4].reset_index(drop=True)

    events_123 = _events_for(df_123_final)
    events_4 = _events_for(df_4_final) if len(df_4_final) > 0 else None

    def _epochs_from(raw):
        epochs_123 = mne.Epochs(raw, events_123, event_id={'NoPinchLog': 0, 'PinchLog': 1},
                                 metadata=df_123_final[meta_cols].reset_index(drop=True),
                                 tmin=0.0, tmax=EPOCH_DURATION, baseline=None,
                                 reject_by_annotation=True, preload=True, on_missing='ignore')
        if events_4 is None:
            return epochs_123
        # reject_by_annotation=False: Step-4 windows sit inside a BAD_BREAK annotation on purpose
        # (see the Step 4 section above) -- MNE's default rejection would otherwise drop all of them.
        epochs_4 = mne.Epochs(raw, events_4, event_id={'NoPinchLog': 0, 'PinchLog': 1},
                               metadata=df_4_final[meta_cols].reset_index(drop=True),
                               tmin=0.0, tmax=EPOCH_DURATION, baseline=None,
                               reject_by_annotation=False, preload=True, on_missing='ignore')
        return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4

    epochs = _epochs_from(eeg)

    # Same windows cut from the decision copy (only used to take the AutoReject decisions). It
    # carries the same annotations, so reject_by_annotation drops exactly the same epochs.
    decision_epochs = None
    if decision_eeg is not None:
        decision_epochs = _epochs_from(decision_eeg)
        assert np.array_equal(decision_epochs.metadata['epoch_id'].to_numpy(), epochs.metadata['epoch_id'].to_numpy()), \
            f"{subject}/{session}: decision epochs do not match the dataset epochs"

    stats = dict(
        subject=subject, session=session,
        step1_placed=len(step1_windows), step1_unfit=step1_unfit,
        step2_placed=len(step2_rows), step2_target=len(step1_windows), step2_exhausted=step2_exhausted,
        step3_candidates=len(step3_rows),
        step4_placed=len(step4_rows), step4_overlapping=len(step4_conflicts),
        step4_dropped_pinch=step4_dropped_pinch,
        total_candidates=len(df), kept_after_reject=len(epochs),
        dropped_by_reject=len(df) - len(epochs), elapsed_s=time.time() - t0,
    )
    if verbose:
        step4_note = (f", step4={stats['step4_placed']} finger-movement tiles "
                       f"({stats['step4_overlapping']} overlap task windows -> is_overlapping=True, "
                       f"{stats['step4_dropped_pinch']} dropped for containing a pinch)"
                       if stats['step4_placed'] + stats['step4_dropped_pinch'] > 0 else "")
        print(f"  {subject}/{session}: step1={stats['step1_placed']} (unfit={stats['step1_unfit']}), "
              f"step2={stats['step2_placed']}/{stats['step2_target']} "
              f"(exhausted={stats['step2_exhausted']}), step3={stats['step3_candidates']}"
              f"{step4_note}, "
              f"kept={stats['kept_after_reject']}/{stats['total_candidates']} "
              f"(dropped by BAD_* rejection: {stats['dropped_by_reject']}), "
              f"{stats['elapsed_s']:.2f}s")
    return epochs, stats, decision_epochs

## Building the full dataset

Since Step 0 avoided redoing ICA (the expensive part) by reusing the already-persisted cleaned
continuous signal, this loop is I/O- and numpy-bound rather than compute-bound. The single shared
`rng` (seeded once, above) is consumed in `SUBJECTS` / per-subject-session order (both sorted,
deterministic), so a full rerun of this notebook -- against an unchanged `data/annotated_dataset`
-- reproduces byte-identical results. (`data/annotated_dataset` is itself regenerated by
`11_preprocessing.ipynb` independently of this notebook and its ICA step is not seeded, so counts
here will drift slightly if that upstream dataset gets regenerated between runs -- that's expected,
not a bug in this notebook.)


In [14]:
t_all = time.time()
ds = {}
ds_decision = {}  # same epochs with an extra AR_DECISION_L_FREQ highpass -- only for the AutoReject decisions, never saved
all_stats = []
for subject in SUBJECTS:
    ds[subject] = {}
    ds_decision[subject] = {}
    for session in SESSIONS_BY_SUBJECT[subject]:
        t_load = time.time()
        eeg, markers = load_session(manifest, subject, session)
        load_s = time.time() - t_load
        decision_eeg = eeg.copy().filter(l_freq=AR_DECISION_L_FREQ, h_freq=None, picks='eeg')
        epochs, stats, decision_epochs = process_session(subject, session, eeg, markers, rng,
                                                         decision_eeg=decision_eeg)
        del decision_eeg
        stats['load_s'] = load_s
        all_stats.append(stats)
        ds[subject][session] = epochs
        ds_decision[subject][session] = decision_epochs

total_sessions = sum(len(v) for v in SESSIONS_BY_SUBJECT.values())
total_wall_s = time.time() - t_all
print(f"\nTotal wall clock for all {total_sessions} sessions across {len(SUBJECTS)} subjects: "
      f"{total_wall_s:.1f}s")

  subject_21/session_1: step1=209 (unfit=0), step2=209/209 (exhausted=False), step3=1305, kept=1430/1723 (dropped by BAD_* rejection: 293), 1.38s
  subject_21/session_2: step1=91 (unfit=0), step2=91/91 (exhausted=False), step3=632, kept=637/814 (dropped by BAD_* rejection: 177), 0.45s
  subject_22/session_1: step1=212 (unfit=0), step2=212/212 (exhausted=False), step3=1082, kept=1047/1506 (dropped by BAD_* rejection: 459), 0.72s
  subject_22/session_2: step1=96 (unfit=0), step2=96/96 (exhausted=False), step3=603, kept=541/795 (dropped by BAD_* rejection: 254), 0.36s
  subject_23/session_1: step1=204 (unfit=0), step2=204/204 (exhausted=False), step3=1102, kept=1077/1510 (dropped by BAD_* rejection: 433), 0.74s
  subject_23/session_2: step1=91 (unfit=0), step2=91/91 (exhausted=False), step3=570, kept=492/752 (dropped by BAD_* rejection: 260), 0.39s
  subject_24/session_1: step1=213 (unfit=0), step2=213/213 (exhausted=False), step3=1247, kept=1337/1673 (dropped by BAD_* rejection: 336), 0.

C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4
C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4


  subject_33/session_1: step1=225 (unfit=0), step2=225/225 (exhausted=False), step3=1320, step4=30 finger-movement tiles (0 overlap task windows -> is_overlapping=True, 0 dropped for containing a pinch), kept=1381/1800 (dropped by BAD_* rejection: 419), 1.30s


C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4
C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4


  subject_33/session_2: step1=115 (unfit=0), step2=115/115 (exhausted=False), step3=619, step4=27 finger-movement tiles (7 overlap task windows -> is_overlapping=True, 3 dropped for containing a pinch), kept=636/876 (dropped by BAD_* rejection: 240), 0.64s


...

C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4
C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4


  subject_39/session_1: step1=218 (unfit=0), step2=218/218 (exhausted=False), step3=1060, step4=29 finger-movement tiles (2 overlap task windows -> is_overlapping=True, 1 dropped for containing a pinch), kept=1138/1525 (dropped by BAD_* rejection: 387), 1.13s


C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4
C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4


  subject_39/session_2: step1=98 (unfit=0), step2=98/98 (exhausted=False), step3=585, step4=30 finger-movement tiles (0 overlap task windows -> is_overlapping=True, 0 dropped for containing a pinch), kept=570/811 (dropped by BAD_* rejection: 241), 0.58s
  subject_40/session_1: step1=114 (unfit=0), step2=89/114 (exhausted=True), step3=475, step4=30 finger-movement tiles (0 overlap task windows -> is_overlapping=True, 0 dropped for containing a pinch), kept=444/708 (dropped by BAD_* rejection: 264), 0.67s

Total wall clock for all 36 sessions across 19 subjects: 37.8s


C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4
C:\Users\nicol\AppData\Local\Temp\ipykernel_34928\1912547794.py:100: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs([epochs_123, epochs_4]) if len(epochs_123) > 0 else epochs_4


## Saving to `data/epoch_dataset_robust/`

Written via `scripts.dataset_manager.write_raw_dataset`, the same helper `11_preprocessing.ipynb`
and `12_data_quality_analysis.ipynb` use -- this mirrors `epoch_dataset_filtered`'s on-disk
convention exactly: one `session-epo.fif` per `subject_X/session_Y/`, plus a root `manifest.json`,
so downstream loading code can use `read_raw_dataset(pathlib.Path('data/epoch_dataset_robust'))`
unchanged.


In [15]:
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from scripts.dataset_manager import write_raw_dataset

write_raw_dataset(ds, OUT_DIR)
print(f"Wrote dataset to {OUT_DIR.resolve()}")


Wrote dataset to C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust


## Summary: per-session build stats

Raw counts from the loop above (candidates placed per step, and how many survived
`reject_by_annotation`).


In [16]:
stats_df = pd.DataFrame(all_stats)
stats_df


,subject,session,step1_placed,step1_unfit,step2_placed,step2_target,step2_exhausted,step3_candidates,step4_placed,step4_overlapping,step4_dropped_pinch,total_candidates,kept_after_reject,dropped_by_reject,elapsed_s,load_s
0,subject_21,session_1,209,0,209,209,False,1305,0,0,0,1723,1430,293,1.380238,0.059169
1,subject_21,session_2,91,0,91,91,False,632,0,0,0,814,637,177,0.454195,0.039062
2,subject_22,session_1,212,0,212,212,False,1082,0,0,0,1506,1047,459,0.716313,0.090268
3,subject_22,session_2,96,0,96,96,False,603,0,0,0,795,541,254,0.357807,0.034515
4,subject_23,session_1,204,0,204,204,False,1102,0,0,0,1510,1077,433,0.742327,0.051178
5,subject_23,session_2,91,0,91,91,False,570,0,0,0,752,492,260,0.386534,0.037606
6,subject_24,session_1,213,0,213,213,False,1247,0,0,0,1673,1337,336,0.815710,0.066568
7,subject_24,session_2,94,0,94,94,False,598,0,0,0,786,584,202,0.663161,0.035198
8,subject_25,session_1,218,0,218,218,False,1218,0,0,0,1654,1329,325,1.176280,0.188120
9,subject_25,session_2,93,0,93,93,False,559,0,0,0,745,538,207,0.477625,0.042773


## Summary: final dataset shape (by label, by `is_overlapping`, per subject)

This is the audit table: the dataset's shape at a glance, without needing to reload it later.


In [18]:
full_md = pd.concat(
    [ds[subject][session].metadata[['subject', 'session', 'PinchType', 'is_overlapping', 'step']]
     for subject in SUBJECTS for session in SESSIONS_BY_SUBJECT[subject]],
    ignore_index=True,
)

print(f"TOTAL epochs in data/epoch_dataset_robust: {len(full_md)}\n")
print("By label (PinchType):")
print(full_md['PinchType'].value_counts(), "\n")
print("By is_overlapping:")
print(full_md['is_overlapping'].value_counts(), "\n")
print("By step (1=jittered pinch, 2=non-overlapping Empty, 3=dense overlapping pool, "
      "4=post-task FingerMovement):")
print(full_md['step'].value_counts(), "\n")
print("Cross tab: label x is_overlapping")
print(pd.crosstab(full_md['PinchType'], full_md['is_overlapping']))


TOTAL epochs in data/epoch_dataset_robust: 33330

By label (PinchType):
PinchType
Empty                 21312
PINCH                  8443
PINCH_WIZARD_OF_OZ     1920
PINCH_IGNORED           787
GAZE                    452
FingerMovement          416
Name: count, dtype: int64 

By is_overlapping:
is_overlapping
True     24580
False     8750
Name: count, dtype: int64 

By step (1=jittered pinch, 2=non-overlapping Empty, 3=dense overlapping pool, 4=post-task FingerMovement):
step
3    24571
2     4181
1     4162
4      416
Name: count, dtype: int64 

Cross tab: label x is_overlapping
is_overlapping      False  True 
PinchType                       
Empty                4181  17131
FingerMovement        407      9
GAZE                  156    296
PINCH                3020   5423
PINCH_IGNORED         303    484
PINCH_WIZARD_OF_OZ    683   1237


## AutoReject on the robust dataset -> `data/epoch_dataset_robust_filtered/`

`data/epoch_dataset_robust` (saved above) is left untouched; this section builds a second, cleaned
copy and writes it to `AR_OUT_DIR`, mirroring how `12_data_quality_analysis.ipynb` derives
`epoch_dataset_filtered` from `epoch_dataset` (same `AutoReject` settings: `n_interpolate=(1, 4, 8)`,
`random_state=42`, one fit per subject/session) -- except for the `consensus` grid.

**`FingerMovement` epochs are never dropped.** They are excluded from AutoReject entirely --
neither used for fitting nor passed through `transform` -- and are concatenated back onto the
cleaned epochs unchanged (no rejection, no interpolation).

Epochs AutoReject marks as bad are dropped; the remaining ones may have up to `n_interpolate`
channels interpolated. Metadata is carried through unchanged, so `epoch_id` stays stable, but
`overlap_neighbor_ids` may now reference epochs that were dropped here.

In [20]:
from autoreject import AutoReject


def autoreject_session(epochs: mne.Epochs, decision_epochs: mne.Epochs):
    '''Fit AutoReject on the non-overlapping epochs of one session's decision copy (extra
    AR_DECISION_L_FREQ highpass), take the drop/interpolate decisions on that copy for every epoch,
    and apply them to the actual (0.3Hz) `epochs`. FingerMovement epochs bypass AutoReject entirely
    and are appended back unchanged, so they are never dropped. Returns (cleaned epochs, per-epoch
    bad mask aligned with `epochs`, number of epochs used for fitting).'''
    epochs = epochs.copy().pick('eeg')
    decision_epochs = decision_epochs.copy().pick('eeg')
    md = epochs.metadata
    finger_mask = (md['PinchType'] == 'FingerMovement').to_numpy()
    ar_epochs = epochs[~finger_mask]
    ar_decision = decision_epochs[~finger_mask]

    clean_mask = ~ar_decision.metadata['is_overlapping'].astype(bool).to_numpy()
    fit_epochs = ar_decision[clean_mask] if clean_mask.sum() >= AR_MIN_FIT_EPOCHS else ar_decision
    ar = AutoReject(n_interpolate=np.array(AR_N_INTERPOLATE), consensus=AR_CONSENSUS,
                    random_state=SEED, n_jobs=-1, verbose=False)
    ar.fit(fit_epochs)
    log = ar.get_reject_log(ar_decision)                       # decisions on the 1Hz copy ...
    ar_clean = ar.transform(ar_epochs, reject_log=log)         # ... applied to the 0.3Hz epochs

    bad = np.zeros(len(epochs), dtype=bool)
    bad[~finger_mask] = log.bad_epochs
    if finger_mask.any():
        epochs_clean = mne.concatenate_epochs([ar_clean, epochs[finger_mask]])
    else:
        epochs_clean = ar_clean
    return epochs_clean, bad, len(fit_epochs)


t_ar = time.time()
ds_ar = {}
ar_rows = []
for subject in SUBJECTS:
    ds_ar[subject] = {}
    for session in SESSIONS_BY_SUBJECT[subject]:
        epochs = ds[subject][session]
        epochs_clean, bad, n_fit = autoreject_session(epochs, ds_decision[subject][session])
        ds_ar[subject][session] = epochs_clean

        md = epochs.metadata.reset_index(drop=True)
        for (label, overlapping), idx in md.groupby(['PinchType', 'is_overlapping'], dropna=False).indices.items():
            ar_rows.append(dict(subject=subject, session=session, PinchType=label,
                                is_overlapping=bool(overlapping), n=len(idx), dropped=int(bad[idx].sum())))
        print(f"  {subject}/{session}: fit on {n_fit} epochs, "
              f"kept={len(epochs_clean)}/{len(epochs)} (dropped by AutoReject: {int(bad.sum())})")

n_finger_in = sum((ds[s][ses].metadata['PinchType'] == 'FingerMovement').sum() for s in ds for ses in ds[s])
n_finger_out = sum((ds_ar[s][ses].metadata['PinchType'] == 'FingerMovement').sum() for s in ds_ar for ses in ds_ar[s])
assert n_finger_in == n_finger_out, f"FingerMovement epochs lost: {n_finger_in} -> {n_finger_out}"
print(f"\nFingerMovement epochs kept: {n_finger_out}/{n_finger_in}")
print(f"AutoReject wall clock: {time.time() - t_ar:.1f}s")

  subject_21/session_1: fit on 347 epochs, kept=1261/1430 (dropped by AutoReject: 169)
  subject_21/session_2: fit on 140 epochs, kept=637/637 (dropped by AutoReject: 0)
  subject_22/session_1: fit on 296 epochs, kept=579/1047 (dropped by AutoReject: 468)
  subject_22/session_2: fit on 129 epochs, kept=312/541 (dropped by AutoReject: 229)
  subject_23/session_1: fit on 293 epochs, kept=962/1077 (dropped by AutoReject: 115)
  subject_23/session_2: fit on 121 epochs, kept=342/492 (dropped by AutoReject: 150)
  subject_24/session_1: fit on 345 epochs, kept=1123/1337 (dropped by AutoReject: 214)
  subject_24/session_2: fit on 142 epochs, kept=532/584 (dropped by AutoReject: 52)
  subject_25/session_1: fit on 351 epochs, kept=1027/1329 (dropped by AutoReject: 302)
  subject_25/session_2: fit on 134 epochs, kept=523/538 (dropped by AutoReject: 15)
  subject_26/session_1: fit on 341 epochs, kept=1129/1316 (dropped by AutoReject: 187)
  subject_26/session_2: fit on 131 epochs, kept=588/599 (dr

In [21]:
write_raw_dataset(ds_ar, AR_OUT_DIR)
print(f"Wrote AutoReject-cleaned dataset to {AR_OUT_DIR.resolve()}")

Wrote AutoReject-cleaned dataset to C:\Users\nicol\github\VIBE\DataAnalysis\data\epoch_dataset_robust_filtered


In [22]:
ar_df = pd.DataFrame(ar_rows)


def drop_rates(by):
    out = ar_df.groupby(by, dropna=False)[['n', 'dropped']].sum()
    out['drop_rate'] = out['dropped'] / out['n']
    return out


print(f"AutoReject dropped {ar_df['dropped'].sum()} of {ar_df['n'].sum()} epochs "
      f"({ar_df['dropped'].sum() / ar_df['n'].sum():.1%})\n")
display(drop_rates(['PinchType', 'is_overlapping']))
display(drop_rates('subject'))

AutoReject dropped 3653 of 33330 epochs (11.0%)



n  dropped  drop_rate
PinchType          is_overlapping                           
Empty              False            4181      538   0.128677
                   True            17131     1993   0.116339
FingerMovement     False             407        0   0.000000
                   True                9        0   0.000000
GAZE               False             156        9   0.057692
                   True              296       18   0.060811
PINCH              False            3020      338   0.111921
                   True             5423      528   0.097363
PINCH_IGNORED      False             303       23   0.075908
                   True              484       35   0.072314
PINCH_WIZARD_OF_OZ False             683       61   0.089312
                   True             1237      110   0.088925

,n,dropped,drop_rate
subject,,,
subject_21,2067,169,0.081761
subject_22,1588,697,0.438917
subject_23,1569,265,0.168897
subject_24,1921,266,0.138470
subject_25,1867,317,0.169791
subject_26,1915,198,0.103394
subject_27,2080,496,0.238462
subject_28,2049,144,0.070278
subject_29,1890,286,0.151323
